In [6]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

zsh:1: command not found: wget


In [7]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [8]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [9]:
print(text[:10])

First Citi


In [10]:
# pip install tiktoken
# not using tiktoken

In [11]:
# create a mapping from characters to integers
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [12]:
# chars

In [14]:
# len(text.)

In [15]:
word = 'charlie munger and buffet are friends'
print(enc.encode(word))

word = 'charlie munger and musk are friends'
print(enc.encode(word))


NameError: name 'enc' is not defined

In [19]:
from torch import tensor
import torch

In [20]:
data = tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:10]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [21]:
train_data = data[:int(0.9*len(data))]
test_data = data[int(0.9*len(data)):]

In [ ]:
# one block is passed to transformer at a time
block_size = 8
train_data[:block_size+1]

tensor([ 7127, 84479,   734, 13036,   581, 18988,  1062,  6544,    11])

In [ ]:
# at a time a batch of 4 "blocks" are processed to better utilize gpu
batch_size = 4
# target = train_data[:block_size]
# output = train_data[1:block_size+1]
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    # x, y = x.to(device), y.to(device)
    return x, y


In [ ]:
x, y = get_batch('train')

In [45]:
from torch.nn import functional as F 
# Batch, time, channels
B,T,C = 4,8,2

wei = torch.zeros((T,T))
tril = torch.tril(torch.ones(T, T))
print(tril)

wei = wei.masked_fill(tril==0, -float('inf'))

wei = F.softmax(wei, dim=1)

print(wei)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


In [50]:
a = tensor([0.1, -0.2, 0.3, 0.2, 0.5])
print(F.softmax(a))
print(a*8)
print(F.softmax(a*8))

tensor([0.1799, 0.1333, 0.2197, 0.1988, 0.2684])
tensor([ 0.8000, -1.6000,  2.4000,  1.6000,  4.0000])
tensor([0.0305, 0.0028, 0.1510, 0.0678, 0.7479])


/var/folders/9b/qmrcn48j35jg224lqm9rzgvm0000gn/T/ipykernel_15677/237591333.py:2: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  print(F.softmax(a))
/var/folders/9b/qmrcn48j35jg224lqm9rzgvm0000gn/T/ipykernel_15677/237591333.py:4: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  print(F.softmax(a*8))
